# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR²](https://senscience.ai/) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset's metadata schema is provided in Croissant JSON-LD format at the URL below.

- Croissant schema: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Install the mlcroissant library if needed
!pip install mlcroissant

## 1. Data Loading
Load the dataset and its metadata using `mlcroissant.Dataset`. This fetches both metadata (fields, record sets) and actual data files referenced in the schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"
# Load Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# View dataset metadata
md = dataset.metadata
print(f"Dataset title: {md.name}\n")
print(f"Dataset description: {md.description}\n")
print(f"Published: {getattr(md, 'datePublished', 'N/A')}")
print(f"Identifier: {getattr(md, 'identifier', 'N/A')}")
print(f"Version: {getattr(md, 'version', 'N/A')}")
print(f"License: {md.license}")

## 2. Data Overview
Let's list all the available record sets and fields, showing their `@id`, label, and description, to help us understand the dataset structure for detailed analysis.

> **Note:** In `mlcroissant`, you reference record sets and fields using their `@id` fields for reliable access.

In [ ]:
# List the available record sets, fields, and columns with their @id

def show_recordsets(md):
    if not hasattr(md, 'recordSets') or not md.recordSets:
        print("No record sets found in the metadata.")
        return []
    rs_ids = []
    print("Available record sets:")
    for rs in md.recordSets:
        print(f"  [@id={rs.id}] Name: {rs.name} - {rs.description or ''}")
        rs_ids.append(rs.id)
        if rs.fields:
            print("    Fields:")
            for f in rs.fields:
                print(f"      [@id={f.id}] Label: {f.label} Type: {getattr(f, 'dataType', 'Unknown')}")
                # If column objects are available
                if getattr(f, 'columns', None):
                    print("        Columns:")
                    for col in f.columns:
                        print(f"        [@id={col.id}] Label: {col.label}")
    return rs_ids

record_set_ids = show_recordsets(md)

## 3. Data Extraction
We'll load data from **each record set** defined in the Croissant metadata into pandas DataFrames. All entities are referenced by their `@id` as recommended.

You may need to consult the printout above to pick a record set of interest for exploration.

In [ ]:
# Automatically extract data from all record sets
dataframes = {}

# Use the first record set if available for demo below
if record_set_ids:
    for rs_id in record_set_ids:
        print(f"Loading records from record set @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded {len(df)} records. Fields: {df.columns.tolist()}\n")
        else:
            print("No records found for this record set.\n")
    # For demonstration, pick the first non-empty dataframe
    selected_rs = next((rs for rs in record_set_ids if rs in dataframes and not dataframes[rs].empty), None)
    if selected_rs:
        print(f"Sample records for record set {selected_rs}:")
        display(dataframes[selected_rs].head())
    else:
        print("No loaded dataframes to show.")
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Let's process and analyze numeric fields in the dataset. We'll demonstrate how to:
- filter records by a threshold,
- normalize a numeric field,
- group by a categorical field.

Be sure to select the correct field `@id` for numeric and grouping columns!

In [ ]:
import numpy as np

# Modify these IDs after inspecting your record set overview output
record_set_id = selected_rs if 'selected_rs' in locals() and selected_rs else None
# Sample: Replace with actual field ids (column names in the dataframe)
numeric_field_id = None  # e.g., 'coverage_percentage' (use the real column name from your schema)
group_field_id = None    # e.g., 'protein_classification' (use the real column name from your schema)

if record_set_id:
    df = dataframes[record_set_id]
    print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
    # Attempt to guess a numeric field if not set
    if not numeric_field_id:
        # Find the first float/int dtype column
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_cols:
            numeric_field_id = numeric_cols[0]
            print(f"Auto-selected numeric field '{numeric_field_id}' for demonstration.")
        else:
            print("No numeric fields available for EDA.")

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records in '{record_set_id}' with {numeric_field_id} > {threshold:.3f}:")
        display(filtered_df.head())
        # Normalize
        filtered_df[numeric_field_id + "_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std()!=0 else 1)
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, numeric_field_id + "_normalized"]].head())
        # Attempt groupby
        if not group_field_id:
            cat_fields = df.select_dtypes(include=["object", "category"]).columns.tolist()
            if cat_fields:
                group_field_id = cat_fields[0]
                print(f"Auto-selected grouping field '{group_field_id}' for demonstration.")
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("Could not identify numeric field for analysis.")
else:
    print("No available record set for EDA.")

## 5. Visualization
Let's visualize the distribution of the selected numeric field, and a relationship between numeric and grouping fields if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only run if there's a valid DataFrame and numeric field
if record_set_id and numeric_field_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    # Box plot by grouping if available
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("Insufficient data for visualization. Please check that a suitable numeric and group field exists.")

## 6. Conclusion
In this notebook, we:
- Loaded FAIR² protein mass spectrometry data via Croissant schema using `mlcroissant`.
- Explored available record sets, fields, and their structure using `@id` references.
- Extracted records and performed basic filtering, normalization, and grouping for EDA.
- Visualized field distributions and relationships between features.

You can extend this workflow for more advanced analyses (e.g., feature engineering, cross-record-set joins, advanced statistical tests) using the rich metadata and data specification from Croissant records.

**Tip:** For large datasets with many record sets or nested structure, always reference entities by their `@id` to keep your analysis robust and reproducible.